[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/07_Jaccard_Coseno.ipynb)


In [ ]:
%pip install scikit-learn pandas

# Jaccard y Coseno: medir qué tan parecidos son dos textos

En el notebook anterior convertimos cada mensaje en un vector numérico con TF-IDF.

Ahora viene la siguiente pregunta natural:

> **¿Cómo medimos qué tan parecidos son dos de esos mensajes?**

Existen dos métricas clásicas para esto:

- **Jaccard**: mide el porcentaje de palabras compartidas entre dos textos
- **Coseno**: mide el ángulo entre los vectores TF-IDF de dos textos

Cada una responde distinto ante las mismas frases. Vamos a verlo con el mismo corpus que ya conocemos.

---

## El corpus

Cuatro mensajes de soporte de una app financiera — los mismos tres de antes, más uno nuevo.

In [ ]:
mensajes = [
    "no puedo iniciar sesión",          # M1
    "no logro acceder a mi cuenta",     # M2 — mismo problema, distintas palabras
    "no puedo hacer la transferencia",  # M3
    "el pago no se procesa",            # M4
]

for i, m in enumerate(mensajes, 1):
    print(f"M{i}: {m}")

---

## Similitud de Jaccard

El índice de Jaccard cuantifica la similitud entre dos conjuntos.

```
                  |A ∩ B|
Jaccard(A, B) = ──────────
                  |A ∪ B|
```

Donde:
- `|A ∩ B|` = número de palabras que **comparten** ambos textos
- `|A ∪ B|` = número total de palabras **distintas** entre los dos textos

El resultado va de 0 (nada en común) a 1 (idénticos).

> Nota: la **cardinalidad** de un conjunto es la cantidad de elementos que tiene. Por eso `|A ∩ B|` es el número de elementos en común y `|A ∪ B|` es el número total de elementos distintos al unir ambos conjuntos.

In [ ]:
def jaccard(str1, str2):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    return len(a & b) / len(a | b)

### Ejemplo básico

Antes de aplicarlo al corpus, veamos la intuición con dos frases simples:

In [ ]:
texto_a = "me gusta mucho el diplomado de la ibero"
texto_b = "me gusta el diplomado"

print(f"A = {set(texto_a.split())}")
print(f"B = {set(texto_b.split())}")
print(f"A ∩ B = {set(texto_a.split()) & set(texto_b.split())}")
print(f"A ∪ B = {set(texto_a.split()) | set(texto_b.split())}")
print()
print(f"Jaccard: {jaccard(texto_a, texto_b):.3f}")

### Aplicado al corpus

Veamos el desglose para M1 y M2, que describen el mismo problema con palabras distintas:

In [ ]:
def jaccard_detalle(str1, str2, label_a="M1", label_b="M2"):
    a = set(str1.lower().split())
    b = set(str2.lower().split())
    inter = a & b
    union = a | b
    score = len(inter) / len(union)
    print(f"  {label_a} = {sorted(a)}")
    print(f"  {label_b} = {sorted(b)}")
    print(f"  A ∩ B = {sorted(inter)}")
    print(f"  A ∪ B = {sorted(union)}")
    print(f"  Jaccard = {len(inter)}/{len(union)} = {score:.3f}")

print(f'M1: "{mensajes[0]}"')
print(f'M2: "{mensajes[1]}"')
print()
jaccard_detalle(mensajes[0], mensajes[1])

In [ ]:
import pandas as pd

n = len(mensajes)
matriz = [[jaccard(mensajes[i], mensajes[j]) for j in range(n)] for i in range(n)]

df_jaccard = pd.DataFrame(
    matriz,
    index=[f"M{i+1}" for i in range(n)],
    columns=[f"M{i+1}" for i in range(n)]
).round(3)

print("Matriz de similitud Jaccard:\n")
print(df_jaccard.to_string())

print("\nMensajes de referencia:")
for i, m in enumerate(mensajes, 1):
    print(f"  M{i}: {m}")

Observa algo importante:

- **M1 vs M2** = valor bajo — aunque ambos son el mismo problema, usan palabras distintas
- **M1 vs M3** = valor más alto — comparten `"no"` y `"puedo"`, aunque hablan de cosas diferentes

Y algo que puede parecer raro: **M1 vs M4 (0.125) es mayor que M1 vs M2 (0.111)**, aunque M4 habla de un pago y M2 del mismo problema que M1. ¿Por qué?

Ambos pares comparten exactamente la misma palabra: `"no"`. La diferencia está en el tamaño de la unión:

- M1 vs M2 → unión de 9 palabras → Jaccard = 1/9 = 0.111
- M1 vs M4 → unión de 8 palabras → Jaccard = 1/8 = 0.125

M2 es más largo, así que la unión es más grande y el resultado es más chico. **Jaccard penaliza los textos largos** aunque digan lo mismo.

Jaccard no entiende el **significado**: premia el vocabulario compartido, no la intención.

---

## Similitud del Coseno

El coseno mide el **ángulo entre dos vectores**. Si dos vectores apuntan en la misma dirección, el ángulo es 0° y el coseno es 1. Si son perpendiculares, el coseno es 0.

```
           A · B
cos(θ) = ─────────
          |A| × |B|
```

Conectamos esto con TF-IDF: cada mensaje ya es un vector (lo vimos en el notebook anterior). Ahora medimos el ángulo entre esos vectores.

Antes de calcular el coseno, veamos cómo luce ese vector numérico para cada mensaje.

Recuerda: TF-IDF convierte cada mensaje en una lista de números — uno por cada palabra del vocabulario. La mayoría son cero (la palabra no aparece en ese mensaje).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

vectorizador = TfidfVectorizer()
X = vectorizador.fit_transform(mensajes)

# Así luce el vector TF-IDF de cada mensaje
terminos = vectorizador.get_feature_names_out()
df_vectores = pd.DataFrame(
    X.toarray(),
    index=[f"M{i+1}" for i in range(len(mensajes))],
    columns=terminos
).round(3)

print("Vector TF-IDF de cada mensaje:")
print(df_vectores.to_string())
print("Cada columna es una palabra del vocabulario.")
print("Cada fila es un mensaje representado como números.")
print("Los ceros indican que esa palabra no aparece en ese mensaje.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# X ya es la matriz TF-IDF que calculamos arriba
# Ahora medimos el coseno entre cada par de vectores
matriz_cos = cosine_similarity(X)

df_coseno = pd.DataFrame(
    matriz_cos,
    index=[f"M{i+1}" for i in range(len(mensajes))],
    columns=[f"M{i+1}" for i in range(len(mensajes))]
).round(3)

print("Matriz de similitud Coseno:")
print(df_coseno.to_string())

print("Mensajes de referencia:")
for i, m in enumerate(mensajes, 1):
    print(f"  M{i}: {m}")

In [ ]:
print("Mensajes de referencia:")
for i, m in enumerate(mensajes, 1):
    print(f"  M{i}: {m}")

# Comparación directa: Jaccard vs Coseno para cada par
print(f"{'Par':<12} {'Jaccard':>10} {'Coseno':>10}  Nota")
print("-" * 68)

pares = [
    (0, 1, "mismo problema, palabras distintas"),
    (0, 2, "problema distinto, comparten 'no puedo'"),
    (0, 3, "problemas distintos"),
    (1, 2, "problemas distintos"),
    (2, 3, "problemas distintos"),
]

for i, j, nota in pares:
    j_score = jaccard(mensajes[i], mensajes[j])
    c_score = float(cosine_similarity(X[i], X[j])[0][0])
    print(f"M{i+1} vs M{j+1}   {j_score:>10.3f} {c_score:>10.3f}  {nota}")

---

## ¿Qué puede y qué no puede cada métrica?

| Métrica | Ventaja | Limitación |
|---|---|---|
| **Jaccard** | Sencilla, rápida, sin dependencias | Trata todas las palabras igual — solo: ¿está o no está? |
| **Coseno + TF-IDF** | Pondera palabras raras más que comunes | Sigue dependiendo del vocabulario exacto |
| **Ambas** | Funcionan bien cuando los textos comparten palabras | No detectan similitud semántica entre vocabularios distintos |

El caso de M1 y M2 lo ilustra bien: `"no puedo iniciar sesión"` y `"no logro acceder a mi cuenta"` significan lo mismo, pero ninguna de las dos métricas lo detecta porque no comparten palabras significativas.

Para resolver eso se necesita representar el **significado** de las palabras, no solo su presencia. A eso le llamamos *embeddings* — tema del siguiente notebook.